In [ ]:
%cd ..

In [1]:
from dotenv import load_dotenv

load_dotenv()


True

In [ ]:
import os
import sys
sys.path.insert(0, r"C:\Users\namtv40\Projects\prefecthq-external-ingestion\ingestions")

import json
import time
import requests
import pandas as pd
import pyarrow as pa
from dateutil import parser
import pyarrow.parquet as pq
from datetime import datetime


In [3]:
# API_KEY = read_hdfs_https(API_KEY_PATH).strip()
# API_COOKIE = read_hdfs_https(API_COOKIE_PATH).strip()
FRESHWORKS_API_KEY = os.getenv("FRESHWORKS_API_KEY")
FRESHWORKS_API_COOKIE = os.getenv("FRESHWORKS_API_COOKIE")

if not FRESHWORKS_API_KEY:
    raise Exception("Not found FRESHWORKS_API_KEY")

if not FRESHWORKS_API_COOKIE:
    raise Exception("Not found FRESHWORKS_API_COOKIE")


RESOURCE_NAME = "deal_pipelines"

if not os.path.exists("./tmp/data"):
    os.makedirs("./tmp/data")


HEADERS = {
    "Authorization": "Token token={}".format(FRESHWORKS_API_KEY),
    "Cookie": FRESHWORKS_API_COOKIE,
    "Content-Type": "application/json",
    "X-Requested-With": "XMLHttpRequest",
}


In [4]:
import trino
from trino.auth import BasicAuthentication
import jaydebeapi

TRINO_URL = os.getenv("TRINO_URL")
TRINO_USER = os.getenv("TRINO_USER")
TRINO_PASSWORD = os.getenv("TRINO_PASSWORD")

def get_existing_deal_ids(sql):
    TRINO_JDBC_JAR = r"C:\Users\namtv40\Libs\trino-jdbc-479.jar"

    jdbc_url = TRINO_URL

    props = {
        "user": TRINO_USER,
        "password": TRINO_PASSWORD,
        "SSL": "false"
    }

    conn = jaydebeapi.connect(
        "io.trino.jdbc.TrinoDriver",
        jdbc_url,
        props,
        TRINO_JDBC_JAR
    )

    cursor = conn.cursor()
    cursor.execute(sql)

    ticket_ids = {str(row[0]) for row in cursor.fetchall()}

    cursor.close()
    conn.close()
    return list(ticket_ids)
    

In [ ]:
from common.config import *
from common.http_util import *
from common.crawler_util import *
from common.ambari_util import *


def fetch_resource_name_freshwork(resource_name, records, **kwargs):
    print("Start crawl : ", resource_name)
    start_time = time.time()

    HDFS_BASE = "s3a://vcs-raw/crm-raw"
    # STATE_PATH = rc["state_path"]
    # BASE_URL = None
    # RESOURCE_URL = None
    # API_KEY_PATH = rc["api_key_path"]
    # API_COOKIE_PATH = rc["api_cookie_path"]
    # QUERY_PARAMS = None
    ENABLE_STATE =False
    HIVE_DB = "crm_raw"
    # crawl_mode = "modified_and_new"
    crawl_mode = kwargs.get("crawl_mode", "modified_and_new")
    schema_local_path = None
    # result_json_key = "deleted_deals"

    print(resource_name)
    start_time = time.time()
    # =========================
    # MAIN
    # =========================
    if ENABLE_STATE:
        last_state = read_last_state(resource_name)
        print("Last state =", last_state)
    else:
        last_state = None

    if not records:
        print("No new data")
        out_of_data = True
        return True

    # =========================
    # Pandas → Parquet
    # =========================

    now = datetime.now()
    partition_path = "{}/{}".format(HDFS_BASE, resource_name)

    filename = "data_{}_{}{:02d}{:02d}_{}{:02d}{:02d}.parquet".format(
        resource_name, now.year, now.month, now.day, now.hour, now.minute, now.second
    )
    local_parquet = "./tmp/data/crm_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_parquet), exist_ok=True)

    # df.to_parquet(local_parquet,engine="pyarrow", compression="snappy", index=False)
    records = convert_json_add_ts_columns(records)

    schema_tm_path = "./resources/parquet_schema/crm_raw/{}.json".format(resource_name)
    schema = None
    if schema_local_path:
        schema = load_pyarrow_schema_from_json(schema_local_path)

    if os.path.exists(schema_tm_path):
        schema = load_pyarrow_schema_from_json(schema_tm_path)

    if not schema:
        schema = infer_schema_from_json(records)
        schema_json = save_pyarrow_type_to_json(schema)
        write_file_json(schema_tm_path, schema_json)

    records = convert_json_list_by_arrow_schema(records, schema)

    df = pd.DataFrame(records)
    data_table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)

    pq.write_table(data_table, local_parquet, compression="snappy")

    if crawl_mode == "static":
        replace_hdfs_https("{}".format(partition_path), local_parquet)
    else:
        upload_hdfs_https("{}".format(partition_path), local_parquet)

    print("Uploaded parquet to", partition_path)

    # =========================
    # Generate SQL (TEXT ONLY)
    # =========================
    sql = gen_spark_create_table(
        schema=schema,
        db=HIVE_DB,
        table=resource_name,
        location="{}/{}".format(HDFS_BASE, resource_name),
    )

    # filename = "create_table_{}_{}{:02d}{:02d}.sql".format(
    #     resource_name,
    #     now.hour,
    #     now.minute,
    #     now.second
    # )

    filename = "create_table_{}.sql".format(resource_name)

    local_sql = "./tmp/data/crm_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_sql), exist_ok=True)

    with open(local_sql, "w") as f:
        f.write(sql)

    # upload_hdfs_https(
    #     "{}/{}".format(HDFS_BASE, resource_name),
    #     local_sql
    # )

    print("Uploaded SQL definition")

    # =========================
    # Save new state
    # =========================
    if ENABLE_STATE and "updated_at" in df.columns:
        max_ts = get_max_updated_at_str(df)
        print("last state ", resource_name, "max_ts=", max_ts)
        max_ts = subtract_minutes(max_ts, 30)
        write_last_state(max_ts, resource_name)
        print("last state ", resource_name, "max_ts=", max_ts)

    elapsed = time.time() - start_time
    print("Loop {} took {:.3f}s".format(resource_name, elapsed))
    if len(records) < 100:
        print("No new data")
        out_of_data = True
        return True
    return False


In [6]:
import time
def check_existed(name, deal_id, url_pattern: str):
    url = url_pattern % (str(deal_id))
    # print(url)
    for i in range(5):
        try:
            # Sử dụng phương thức HEAD thay vì GET
            response = requests.head(url, headers=HEADERS,timeout=REQUEST_TIMEOUT,
                proxies=PROXIES if USE_PROXY else None,)
            
            if response.status_code == 404:
                print(f"{name} {deal_id} đã bị xóa hoặc không tồn tại.")
                return False
            elif response.status_code == 200:
                print(f"{name} {deal_id} vẫn tồn tại.")
                return True
            else:
                print(f"{name} Lỗi khác: {response.status_code}")
                time.sleep(5)
                continue
        except Exception as e:
            print(f"{name} Đã xảy ra lỗi kết nối: {e}")
            time.sleep(5)
            continue

In [7]:
url_templates = [
    {
        "name":"deleted_deals",
        "link":"https://crm.viettelsecurity.com/crm/sales/api/deals/%s?include=id",
        "id_sql":"select distinct id from hive.crm_raw.deals",
    },
    
    {
        "name":"deleted_cm_contracts",
        "link":"https://crm.viettelsecurity.com/crm/sales/api/custom_module/cm_contracts/%s",
        "id_sql":"select distinct id from hive.crm_raw.cm_contracts",
    },
    
    {
        "name":"deleted_sales_accounts",
        "link":"https://crm.viettelsecurity.com/crm/sales/api/sales_accounts/%s",
        "id_sql":"select distinct id from hive.crm_raw.sales_accounts",
    },
    
    {
        "name":"deleted_st_users",
        "link":"https://crm.viettelsecurity.com/crm/sales/api/settings/users/%s",
        "id_sql":"select distinct id from hive.crm_raw.st_users",
    },
    {
        "name":"deleted_contacts",
        "link":"https://crm.viettelsecurity.com/crm/sales/api/contacts/%s",
        "id_sql":"select distinct id from hive.crm_raw.contacts",
    },
    {
        "name":"deleted_cm_pricebook",
        "link":"https://crm.viettelsecurity.com/crm/sales/api/custom_module/cm_pricebook/%s",
        "id_sql":"select distinct id from hive.crm_raw.cm_pricebook",
    },
    {
        "name":"deleted_cm_catalog",
        "link":"https://crm.viettelsecurity.com/crm/sales/api/custom_module/cm_catalog/%s",
        "id_sql":"select distinct id from hive.crm_raw.cm_catalog",
    }
]

In [ ]:
def check_many_ids(name: str,id_sql: str, link:str ):
    # name = "deleted_deals"
    # id_sql = "select distinct id from hive.crm_raw.deals"
    ids = get_existing_deal_ids(id_sql)
    print(len(ids))
    deleted_ids = []
    none_ids = []
    for idx,id in enumerate(ids):
        existed = check_existed(name, id, link)
        if existed  == False:
            deleted_ids += [{"id":id}]
        elif existed is None:
            none_ids += [id]
            
    print(name, "deleted_ids: ",len(deleted_ids))
    print(name, "none_ids: ",len(none_ids))
    
    with open("./tmp/crm_raw/deleted_records/"+name+"_deleted_ids.json", mode="wt", encoding='utf-8') as f:
        f.write(json.dumps(deleted_ids, ensure_ascii=False))
        
    with open("./tmp/crm_raw/deleted_records/"+name+"_none_ids.json", mode="wt", encoding='utf-8') as f:
        f.write(json.dumps(none_ids, ensure_ascii=False))
        
    fetch_resource_name_freshwork(name, deleted_ids, crawl_mode="static")

In [9]:
for url_template in url_templates:
    name = url_template.get("name")
    id_sql = url_template.get("id_sql")
    link = url_template.get("link")
    check_many_ids(name,id_sql, link)

1531
deleted_deals 50001725873 vẫn tồn tại.
deleted_deals 50001723395 vẫn tồn tại.
deleted_deals 50001829058 vẫn tồn tại.
deleted_deals 50001753634 vẫn tồn tại.
deleted_deals 50001838218 vẫn tồn tại.
deleted_deals 50001813717 vẫn tồn tại.
deleted_deals 50001821349 vẫn tồn tại.
deleted_deals 50001827670 vẫn tồn tại.
deleted_deals 50001830054 vẫn tồn tại.
deleted_deals 50001836780 vẫn tồn tại.
deleted_deals 50001836784 vẫn tồn tại.
deleted_deals 50001741280 vẫn tồn tại.
deleted_deals 50001723363 vẫn tồn tại.
deleted_deals 50001839436 vẫn tồn tại.
deleted_deals 50001814023 vẫn tồn tại.
deleted_deals 50001724544 vẫn tồn tại.
deleted_deals 50001761803 vẫn tồn tại.
deleted_deals 50001837443 vẫn tồn tại.
deleted_deals 50001814392 vẫn tồn tại.
deleted_deals 50001837466 vẫn tồn tại.
deleted_deals 50001814044 vẫn tồn tại.
deleted_deals 50001838219 vẫn tồn tại.
deleted_deals 50001724775 vẫn tồn tại.
deleted_deals 50001768465 vẫn tồn tại.
deleted_deals 50001828314 vẫn tồn tại.
deleted_deals 500018

FileNotFoundError: [Errno 2] No such file or directory: './tmp/deleted_records/deleted_deals_deleted_ids.json'

In [ ]:
# with open("./tmp/deleted_records/deleted_sales_accounts_deleted_ids.json", mode="rt", encoding='utf-8') as f:
#     records = json.load(f)
#     fetch_resource_name_freshwork("deleted_sales_accounts", records, crawl_mode="modified_and_new")